In [ ]:
# ================================
# 2_attention_unet_training.ipynb
# ================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

# -----------------
# Attention U-Net
# -----------------
class SeparableConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, padding=0, stride=1, bias=True):
        super().__init__()
        self.depth_wise = nn.Conv2d(in_channels, in_channels, kernel_size=kernel_size,
                                    padding=padding, stride=stride, bias=bias, groups=in_channels)
        self.point_wise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=bias)
    def forward(self, x):
        x = self.depth_wise(x)
        x = self.point_wise(x)
        return x

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1):
        super().__init__()
        self.conv1 = SeparableConv2d(in_channels, out_channels, kernel_size=kernel_size, padding=1, stride=stride, bias=True)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.act1 = nn.ReLU(inplace=True)

        self.conv2 = SeparableConv2d(out_channels, out_channels, kernel_size=kernel_size, padding=1, stride=stride, bias=True)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.act2 = nn.ReLU(inplace=True)

        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                SeparableConv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        residual = x
        x = self.act1(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        if self.downsample:
            residual = self.downsample(residual)
        x += residual
        return self.act2(x)

class AttentionBlock(nn.Module):
    def __init__(self, in_channels, gating_channels, inter_channels):
        super().__init__()
        self.w_g = nn.Sequential(
            nn.Conv2d(gating_channels, inter_channels, kernel_size=1),
            nn.BatchNorm2d(inter_channels)
        )
        self.w_x = nn.Sequential(
            nn.Conv2d(in_channels, inter_channels, kernel_size=1),
            nn.BatchNorm2d(inter_channels)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(inter_channels, 1, kernel_size=1),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

    def forward(self, g, x):
        g1 = self.w_g(g)
        x1 = self.w_x(x)
        psi = F.relu(x1 + g1)
        psi = self.psi(psi)
        return x * psi

class AttentionUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.pool = nn.MaxPool2d(2, 2)
        self.encoder1 = ResidualBlock(in_channels, 64)
        self.encoder2 = ResidualBlock(64, 128)
        self.encoder3 = ResidualBlock(128, 256)
        self.encoder4 = ResidualBlock(256, 512)
        self.encoder5 = ResidualBlock(512, 1024)

        self.up5 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.att5 = AttentionBlock(512, 512, 256)
        self.upconv5 = ResidualBlock(1024, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.att4 = AttentionBlock(256, 256, 128)
        self.upconv4 = ResidualBlock(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.att3 = AttentionBlock(128, 128, 64)
        self.upconv3 = ResidualBlock(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.att2 = AttentionBlock(64, 64, 32)
        self.upconv2 = ResidualBlock(128, 64)

        self.outconv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.encoder1(x)
        x2 = self.encoder2(self.pool(x1))
        x3 = self.encoder3(self.pool(x2))
        x4 = self.encoder4(self.pool(x3))
        x5 = self.encoder5(self.pool(x4))

        d5 = self.att5(self.up5(x5), x4)
        d5 = self.upconv5(torch.cat((d5, self.up5(x5)), 1))

        d4 = self.att4(self.up4(d5), x3)
        d4 = self.upconv4(torch.cat((d4, self.up4(d5)), 1))

        d3 = self.att3(self.up3(d4), x2)
        d3 = self.upconv3(torch.cat((d3, self.up3(d4)), 1))

        d2 = self.att2(self.up2(d3), x1)
        d2 = self.upconv2(torch.cat((d2, self.up2(d3)), 1))

        return self.outconv(d2)

# -----------------
# Discriminator + Loss
# -----------------
class BasicBlock(nn.Module):
    def __init__(self, inplanes, outplanes, kernel_size=4, stride=2, padding=1, norm=True):
        super().__init__()
        self.conv = nn.Conv2d(inplanes, outplanes, kernel_size, stride, padding)
        self.isn = nn.InstanceNorm2d(outplanes) if norm else None
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)
    def forward(self, x):
        x = self.conv(x)
        if self.isn is not None:
            x = self.isn(x)
        return self.lrelu(x)

class ConditionalDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = BasicBlock(2, 64, norm=False)
        self.block2 = BasicBlock(64, 128)
        self.block3 = BasicBlock(128, 256)
        self.block4 = BasicBlock(256, 512)
        self.block5 = nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=1)
    def forward(self, x, cond):
        x = torch.cat([x, cond], dim=1)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        return self.block5(x)

class GeneratorLoss(nn.Module):
    def __init__(self, alpha=100):
        super().__init__()
        self.alpha = alpha
        self.bce = nn.BCEWithLogitsLoss()
        self.l1 = nn.L1Loss()
    def forward(self, fake, real, fake_pred):
        target = torch.ones_like(fake_pred)
        return self.bce(fake_pred, target) + self.alpha * self.l1(fake, real)

class DiscriminatorLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, fake_pred, real_pred):
        fake_t = torch.zeros_like(fake_pred)
        real_t = torch.ones_like(real_pred)
        return (self.bce(fake_pred, fake_t) + self.bce(real_pred, real_t)) / 2

# -----------------
# Training Setup
# -----------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
generator = AttentionUNet().to(device)
discriminator = ConditionalDiscriminator().to(device)

g_optimizer = torch.optim.Adam(generator.parameters(), lr=3e-4, betas=(0.5, 0.999))
d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=3e-4, betas=(0.5, 0.999))
g_loss_fn, d_loss_fn = GeneratorLoss(), DiscriminatorLoss()

EPOCHS = 100
history = {'gen_loss': [], 'dis_loss': []}

for epoch in range(EPOCHS):
    generator.train(), discriminator.train()
    gen_loss_sum, dis_loss_sum = 0, 0
    for im, _, masked in tqdm(dataloaders['train']):
        im, masked = im.to(device), masked.to(device)

        # ---- Train Discriminator ----
        d_optimizer.zero_grad()
        real_pred = discriminator(im, masked)
        fake = generator(masked).detach()
        fake_pred = discriminator(fake, masked)
        d_loss = d_loss_fn(fake_pred, real_pred)
        d_loss.backward()
        d_optimizer.step()

        # ---- Train Generator ----
        g_optimizer.zero_grad()
        fake = generator(masked)
        fake_pred = discriminator(fake, masked)
        g_loss = g_loss_fn(fake, im, fake_pred)
        g_loss.backward()
        g_optimizer.step()

        gen_loss_sum += g_loss.item()
        dis_loss_sum += d_loss.item()

    history['gen_loss'].append(gen_loss_sum / len(dataloaders['train']))
    history['dis_loss'].append(dis_loss_sum / len(dataloaders['train']))

    print(f"Epoch [{epoch+1}/{EPOCHS}] - Gen Loss: {history['gen_loss'][-1]:.4f}, Dis Loss: {history['dis_loss'][-1]:.4f}")

# Save model weights
torch.save(generator.state_dict(), "generator_wt.pt")
